### **Exercício 1: Ajustamento de Rede Altimétrica**  
**Objetivo:** Determinar as altitudes de pontos \(A\), \(B\), e \(C\) usando mínimos quadrados.  

**Dados:**  
- Observações (diferenças de nível):  
  - $\Delta H_{AB} = +2{,}540 \, \text{m}\ $
  - $\Delta H_{BA} = -2{,}545 \, \text{m}\ $
  - $\Delta H_{AC} = +1{,}200 \, \text{m}\ $  
  - $\Delta H_{CB} = +1{,}335 \, \text{m}\ $

- Peso das observações: $p_i = 1$ (todas igualmente precisas).  

**Tarefas:**  
1. Monte a matriz \(A\) e o vetor \(L\) do modelo paramétrico.  
2. Resolva o sistema $\hat{X} = (A^T P A)^{-1} A^T P L$  (fixando $H_A = 100{,}000 \, \text{m}$).  
3. Calcule os resíduos e a variância a posteriori.  

In [26]:
import numpy as np

# 1. Dados de entrada (Lb)
Lb = np.array([[2.540], [-2.545], [1.200], [1.335]])
P = np.eye(4)
HA = 100.000







No modelo linear $L = AX$, onde $L = L_b - L_0$, se você escolher valores iniciais $X_0$ que já são as altitudes aproximadas, o ajuste deve ser direto.

1. Definição do ModeloFixando $H_A = 100,000$ m, nossas incógnitas são $X = [H_B, H_C]^T$.As equações de observação são:

$H_B - H_A = \Delta H_{AB} \Rightarrow H_B - 100,000 = 2,540$

$H_A - H_B = \Delta H_{BA} \Rightarrow 100,000 - H_B = -2,545$

$H_C - H_A = \Delta H_{AC} \Rightarrow H_C - 100,000 = 1,200$

$H_B - H_C = \Delta H_{CB} \Rightarrow H_B - H_C = -1,335$

In [27]:
# 2. Valores iniciais (Chute inicial X0)
# Vamos usar valores propositalmente afastados para mostrar a convergência
Xa = np.array([[105.0], [105.0]])

# Critérios de parada
tolerancia = 0.0001
max_iter = 10
delta_x = np.array([[1.0], [1.0]])
it = 0

print(f"{'It':<5} | {'HB':<10} | {'HC':<10} | {'Max Delta':<10}")
print("-" * 45)

while np.max(np.abs(delta_x)) > tolerancia and it < max_iter:
    HB, HC = Xa[0,0], Xa[1,0]

    # 3. Vetor de observações calculadas L0=f(Xa)
    # Baseado nas equações: HB-HA, HA-HB, HC-HA, HB-HC
    L0 = np.array([
        [HB - HA],
        [HA - HB],
        [HC - HA],
        [HB - HC]
    ])

    # 4. Vetor L (L = Lb - L0)
    L = Lb - L0

    # 5. Matriz Jacobiana A (Derivadas parciais)
    # d/dHB  d/dHC
    A = np.array([
        [ 1,  0],  # d(HB-HA)/dHB = 1
        [-1,  0],  # d(HA-HB)/dHB = -1
        [ 0,  1],  # d(HC-HA)/dHC = 1
        [ 1, -1]   # d(HB-HC)
    ])

    # 6. Sistema Normal: N * delta_x = U
    N = A.T @ P @ A
    U = A.T @ P @ L
    delta_x = np.linalg.inv(N) @ U

    # 7. Atualização dos parâmetros
    Xa = Xa + delta_x
    it += 1

    print(f"{it:<5} | {Xa[0,0]:.4f} | {Xa[1,0]:.4f} | {np.max(np.abs(delta_x)):.6f}")

# 8. Resultados Finais
V = A @ delta_x - L # V = Ax - L no modelo linearizado
vtpv = V.T @ P @ V
sigma_post = vtpv / (4 - 2)

print("-" * 45)
print(f"Altitude Final HB: {Xa[0,0]:.4f} m")
print(f"Altitude Final HC: {Xa[1,0]:.4f} m")
print(f"Variância a posteriori: {sigma_post[0,0]:.8f}")
print(f"Número de iterações:{it}")

It    | HB         | HC         | Max Delta 
---------------------------------------------
1     | 102.5410 | 101.2030 | 3.797000
2     | 102.5410 | 101.2030 | 0.000000
---------------------------------------------
Altitude Final HB: 102.5410 m
Altitude Final HC: 101.2030 m
Variância a posteriori: 0.00001750
Número de iterações:2


### **Exercício 2: Ajustamento de uma Rede GNSS**  
**Objetivo:** Estimar as coordenadas $(X, Y, Z)$ do ponto $Q$ a partir de observações de baselines GNSS, usando o método paramétrico linear $(L + v = A X_a)$.  

#### **Dados:**  
- **Pontos Fixos (Coordenadas Conhecidas):**  
  - $R_1(X_1 = 1000.000 \, \text{m}, Y_1 = 2000.000 \, \text{m}, Z_1 = 3000.000 \, \text{m})$  
  - $R_2(X_2 = 1100.000 \, \text{m}, Y_2 = 2100.000 \, \text{m}, Z_2 = 3100.000 \, \text{m})$  

- **Observações (Baselines GNSS):**  
  - Baseline $R_1 \rightarrow Q$:  
    $\Delta X_{1Q} = +50.010 \, \text{m}$, $\Delta Y_{1Q} = +60.005 \, \text{m}$, $\Delta Z_{1Q} = +70.008 \, \text{m}$  
  - Baseline $R_2 \rightarrow Q$:  
    $\Delta X_{2Q} = -40.020 \, \text{m}$, $\Delta Y_{2Q} = -30.015 \, \text{m}$, $\Delta Z_{2Q} = -20.010 \, \text{m}$  

- **Matriz de Pesos ($P$):**  
  $$  
  P = I_6 \quad \text{(Matriz identidade 6x6, pois todas as observações têm mesma precisão)}  
  $$  

#### **Modelo Paramétrico:**  
As equações são:  
$$  
\begin{cases}  
\Delta X_{1Q} + v_1 = X_Q - X_1 \\  
\Delta Y_{1Q} + v_2 = Y_Q - Y_1 \\  
\Delta Z_{1Q} + v_3 = Z_Q - Z_1 \\  
\Delta X_{2Q} + v_4 = X_Q - X_2 \\  
\Delta Y_{2Q} + v_5 = Y_Q - Y_2 \\  
\Delta Z_{2Q} + v_6 = Z_Q - Z_2 \\  
\end{cases}  
$$  

**Forma Matricial:**  
$$  
\underbrace{  
\begin{bmatrix}  
\Delta X_{1Q} \\  
\Delta Y_{1Q} \\  
\Delta Z_{1Q} \\  
\Delta X_{2Q} \\  
\Delta Y_{2Q} \\  
\Delta Z_{2Q} \\  
\end{bmatrix}  
}_{L}  
+  
v  
=  
\underbrace{  
\begin{bmatrix}  
1 & 0 & 0 \\  
0 & 1 & 0 \\  
0 & 0 & 1 \\  
1 & 0 & 0 \\  
0 & 1 & 0 \\  
0 & 0 & 1 \\  
\end{bmatrix}  
}_{A}  
\underbrace{  
\begin{bmatrix}  
X_Q \\  
Y_Q \\  
Z_Q \\  
\end{bmatrix}  
}_{X_a}  
-  
\underbrace{  
\begin{bmatrix}  
X_1 \\  
Y_1 \\  
Z_1 \\  
X_2 \\  
Y_2 \\  
Z_2 \\  
\end{bmatrix}  
}_{L_0}  
$$  

#### **Tarefas:**  
1. Monte a matriz $A$ ($6 \times 3$) e o vetor $L_{\text{ajustado}} = L + L_0$.  
2. Resolva o sistema $\hat{X}_a = (A^T P A)^{-1} A^T P L_{\text{ajustado}}$.  
3. Calcule os resíduos $v = A \hat{X}_a - L_{\text{ajustado}}$ e a variância a posteriori $\sigma_0^2 = \frac{v^T P v}{6-3}$.  


### **📌 Exercício 3: Ajustamento de Rede Altimétrica com Múltiplos Vértices**  
#### **🎯 Objetivo**  
Determinar as altitudes ajustadas dos vértices $B$, $C$, $D$ e $E$ a partir de observações de diferenças de nível, utilizando o **método paramétrico linear** ($L + v = A X_a$).

#### **📊 Dados da Rede**  
| **Vértice Fixo** | **Altitude (m)** | **Vértices Desconhecidos** | **Observações (ΔH)** | **Valor (m)** | **Desvio Padrão (mm)** |
|------------------|-----------------|---------------------------|----------------------|--------------|-----------------------|
| $A$              | 100.000         | $A \rightarrow B$         | $\Delta H_{AB}$      | +2.540       | ±1.5                  |
|                  |                 | $B \rightarrow C$         | $\Delta H_{BC}$      | -1.824       | ±2.0                  |
|                  |                 | $C \rightarrow D$         | $\Delta H_{CD}$      | +2.553       | ±1.0                  |
|                  |                 | $A \rightarrow D$         | $\Delta H_{AD}$      | +3.261       | ±1.2                  |
|                  |                 | $D \rightarrow E$         | $\Delta H_{DE}$      | -0.052       | ±1.8                  |
|                  |                 | $B \rightarrow E$         | $\Delta H_{BE}$      | -0.129       | ±1.5                  |

**Matriz de Pesos ($P$):**  
$$
P = \text{diag}\left(\frac{1}{\sigma_i^2}\right) = \begin{bmatrix}
\frac{1}{1.5^2} & 0 & \cdots & 0 \\
0 & \frac{1}{2.0^2} & \cdots & 0 \\
\vdots & \vdots & \ddots & \vdots \\
0 & 0 & \cdots & \frac{1}{1.5^2}
\end{bmatrix}
$$

#### **📝 Modelo Paramétrico**  
**Equações de Observação:**  
$$
\begin{cases}
\Delta H_{AB} + v_1 = H_B - H_A \\
\Delta H_{BC} + v_2 = H_C - H_B \\
\Delta H_{CD} + v_3 = H_D - H_C \\
\Delta H_{AD} + v_4 = H_D - H_A \\
\Delta H_{DE} + v_5 = H_E - H_D \\
\Delta H_{BE} + v_6 = H_E - H_B \\
\end{cases}
$$

**Forma Matricial:**  
$$
\underbrace{
\begin{bmatrix}
2.540 \\ -1.820 \\ 0.750 \\ 3.290 \\ -0.450 \\ -3.010
\end{bmatrix}
}_{L}
+
v
=
\underbrace{
\begin{bmatrix}
-1 & 1 & 0 & 0 \\
0 & -1 & 1 & 0 \\
0 & 0 & -1 & 1 \\
-1 & 0 & 0 & 1 \\
0 & 0 & -1 & 1 \\
0 & -1 & 0 & 1 \\
\end{bmatrix}
}_{A}
\underbrace{
\begin{bmatrix}
H_B \\ H_C \\ H_D \\ H_E
\end{bmatrix}
}_{X_a}
-
\underbrace{
\begin{bmatrix}
100.000 \\ 0 \\ 0 \\ 100.000 \\ 0 \\ 0
\end{bmatrix}
}_{L_0}
$$

#### **✅ Tarefas**  
1. **Montar o sistema** $L + v = A X_a - L_0$ (com $H_A = 100.000\, \text{m}$ fixo).  
2. **Resolver** o ajustamento por mínimos quadrados:  
   - Calcule $\hat{X}_a = (A^T P A)^{-1} A^T P L_{\text{ajustado}}$.  
3. **Avaliação de precisão:**  
   - Calcule os **resíduos** ($v$) e a **variância a posteriori** ($\sigma_0^2 = \frac{v^T P v}{6-4}$).  

